# Scenario 19 — Cross-Agent Distributed Tracing: Different Keys, One Trace

> *"How will end-to-end agentic workflows be traceable when the agents
> are running as separate processes and generating logs under different
> API keys? Will be beneficial to have a demo."*
> — Customer ask

## The Question, In One Line

Can two microservices, authenticated with **different Galileo API keys**
(but in the **same Galileo org**), still produce **one unified trace
tree** in **one log stream**?

**Answer: yes — and this notebook proves it end to end in under a minute.**

## The Production Shape We're Mirroring

| Component | Owned by | Galileo key |
|---|---|---|
| Orchestrator service | Team A | `KEY_A` |
| Retrieval service | Team B | `KEY_B` |
| Observability project | Platform team | *both keys have write access* |

Each service deploys with its **own credentials** (security wins). Every
distributed trace lands as **one row** in **one log stream** that the
SRE on call opens (operability wins).

## What You'll See, In Five Steps

1. Two FastAPI agents launched as **real OS subprocesses**, each loading
   its own `GALILEO_API_KEY` from `.env`.
2. Both subprocesses point at the **same** project and log stream.
3. The orchestrator answers a question by HTTP-calling the retrieval agent.
4. Galileo's tracing headers (`X-Galileo-Trace-ID`, `X-Galileo-Parent-ID`)
   propagate the trace_id across the two processes.
5. In the Galileo console, the full distributed trace —
   `answer → analyze → retrieve → search → openai` — appears as **one
   nested trace** in the shared log stream.

> **Reference:** extends [sdk-examples / python / logging-samples /
> distributed-tracing](https://github.com/rungalileo/sdk-examples/tree/main/python/logging-samples/distributed-tracing).
> Notebook 18 = single key. Notebook 19 = two keys, one org.

> **Sidebar — for a different org topology.** If services were
> authenticated by keys in *different* Galileo orgs (true multi-tenant),
> spans couldn't physically share a project — the pattern there is
> OpenTelemetry → shared collector → per-org fan-out (notebooks 10 / 11 /
> 12). The HTTP-header trace propagation shown here is still the
> foundation in both cases.


## Step 1 — Each agent loads its own credentials

The notebook reads four values from `.env`:

| Variable | Used by | Resolution |
|---|---|---|
| `GALILEO_API_KEY`    | Orchestrator | required |
| `GALILEO_API_KEY2`   | Retrieval    | falls back to `GALILEO_API_KEY` if absent |
| `GALILEO_PROJECT`    | **Both** agents | defaults to `GalileoEval_S19_CrossAgent` |
| `GALILEO_LOG_STREAM` | **Both** agents | defaults to `cross-agent-trace` |

**Credential isolation lives at the key layer. Observability consolidation
lives at the project layer.** Each team rotates its own key independently;
SREs still see one unified view. That separation is the whole point of
the pattern.

> If `GALILEO_API_KEY2` isn't set, the demo still runs but degrades to
> "same key, same project" — fine for exercising the mechanics, but it
> doesn't tell the credential-isolation story. The cell below prints
> which mode you're in.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in repo root or notebooks/')

# Per-agent keys: orchestrator uses the primary key; retrieval uses the
# secondary if present, else falls back to the primary so the demo still runs.
ORCH_API_KEY = os.environ.get('GALILEO_API_KEY')
RET_API_KEY  = os.environ.get('GALILEO_API_KEY2') or ORCH_API_KEY

# One shared project + log stream for BOTH agents — the consolidated view.
SHARED_PROJECT    = os.environ.get('GALILEO_PROJECT', 'GalileoEval_S19_CrossAgent')
SHARED_LOG_STREAM = os.environ.get('GALILEO_LOG_STREAM', 'cross-agent-trace')

CONSOLE_URL = os.environ.get('GALILEO_CONSOLE_URL', 'https://app.galileo.ai')

if not ORCH_API_KEY:
    raise RuntimeError('Need GALILEO_API_KEY in .env.')

DISTINCT_KEYS = ORCH_API_KEY != RET_API_KEY

print(f'.env: {ENV_FILE}')
print(f'console: {CONSOLE_URL}')
print(f'shared project    : {SHARED_PROJECT}')
print(f'shared log stream : {SHARED_LOG_STREAM}')
print(f'orchestrator key  : …{ORCH_API_KEY[-6:]}')
print(f'retrieval    key  : …{RET_API_KEY[-6:]}')
print(f'mode: {"DISTINCT KEYS, SHARED PROJECT (target demo)" if DISTINCT_KEYS else "SAME KEY (no GALILEO_API_KEY2 set — demo degraded)"}')

.env: /Users/binliu/Dev/rungalileo/demos/galileo-test/.env
console: https://console.demo-v2.galileocloud.io/
shared project    : GalileoEval_S19_CrossAgent
shared log stream : cross-agent-trace
orchestrator key  : …-NHDSE
retrieval    key  : …xcBRDY
mode: DISTINCT KEYS, SHARED PROJECT (target demo)


## Step 2 — Define the two agent services (inline, no files written)

Each agent is a tiny FastAPI service. Instead of writing them to a
separate directory on disk, we keep the source as **Python strings right
here in the notebook** (`RETRIEVAL_SRC`, `ORCHESTRATOR_SRC`). Step 3
hands those strings to `python -c <SOURCE>` to launch each as a
subprocess — so the notebook is the single source of truth, no `/tmp`
roundtrip, and editing a cell is enough to redeploy.

Both agents follow the same Galileo recipe:

1. **`galileo_context.init(...)`** at module load — binds the process to
   its project + log stream.
2. **`TracingMiddleware`** on the FastAPI app — accepts inbound
   `X-Galileo-Trace-ID` / `X-Galileo-Parent-ID` headers and resumes the
   parent trace.
3. **`@log` decorators** on handler functions — every call becomes a span.
4. **`FlushAfterRequestMiddleware`** — blocks the response until every
   span for the request is confirmed ingested by the Galileo backend.
   *Why it's there:* in `mode="distributed"` the SDK submits each span
   ingest as a background task and the handler returns immediately — so
   in a short-lived demo, in-flight ingests can get cut off when the
   subprocess terminates. Adding this middleware makes the demo
   deterministic: every trace shows every span, every time.

They differ in just two places:

- **Retrieval agent** — exposes `POST /retrieve`. The core lookup uses
  `@log(span_type='retriever')` so Galileo activates RAG-specific metrics.
- **Orchestrator agent** — exposes `POST /answer`. Inside it: analyze →
  call retrieval over HTTP (with `get_tracing_headers()`) → call OpenAI →
  return the answer plus the current `trace_id`.

Each source ends with `uvicorn.run(app, ...)` so it's a runnable script,
not a library. Port comes from the `AGENT_PORT` env var that Step 3 sets
per subprocess.

> **Production note:** long-running services usually don't need
> `FlushAfterRequestMiddleware` — graceful shutdown of the process
> drains in-flight ingest tasks via the SDK's `atexit` handler. The
> middleware is a demo-time guarantee, not a production requirement.


In [2]:
import textwrap

RETRIEVAL_PORT = 8775
ORCHESTRATOR_PORT = 8776

# Shared middleware that both agents install. Why this exists:
# In `mode="distributed"`, every span ingest is submitted as a background task on
# the logger's ThreadPoolExecutor and the function returns immediately — i.e. the
# HTTP response can be sent BEFORE the span has actually been ingested by the
# Galileo backend. If the subprocess is then SIGTERM/SIGKILLed (e.g. by the
# cleanup cell), any in-flight ingest tasks are lost, and only some of the
# trace's spans land in the console. By calling `galileo_context.flush()` AFTER
# `call_next` returns (which is after @log's finalize has run), we block the
# response until every span for this request is confirmed in the backend.
FLUSH_MIDDLEWARE_SRC = textwrap.dedent('''
    from starlette.middleware.base import BaseHTTPMiddleware

    class FlushAfterRequestMiddleware(BaseHTTPMiddleware):
        """Blocks the response until all Galileo spans for this request are ingested.

        Must be added BEFORE TracingMiddleware so that it ends up INSIDE
        TracingMiddleware in the dispatch chain — i.e. flush() runs while the
        trace_id / parent_id contextvars are still set, so get_logger_instance()
        finds the right per-request logger.
        """
        async def dispatch(self, request, call_next):
            response = await call_next(request)
            try:
                galileo_context.flush()
            except Exception as e:
                import sys
                print(f"galileo flush failed: {e}", file=sys.stderr, flush=True)
            return response
''').lstrip()

# Each agent source is a self-contained Python script that:
# 1. Reads its project / log_stream / port / API key from the environment that
#    Step 3 will inject when it launches the subprocess.
# 2. Defines a FastAPI app with our flush middleware + Galileo TracingMiddleware.
# 3. Calls `uvicorn.run(app, ...)` at the bottom — that's the entry point when
#    we invoke this source via `python -c <RETRIEVAL_SRC>` in Step 3.
RETRIEVAL_SRC = textwrap.dedent('''
    """Retrieval agent — its OWN Galileo project & key."""
    import os
    from dotenv import load_dotenv
    from fastapi import FastAPI
    from pydantic import BaseModel

    load_dotenv()  # pick up OPENAI_API_KEY etc; agent-specific Galileo vars are injected via subprocess env.

    from galileo import galileo_context, log
    from galileo.middleware.tracing import TracingMiddleware

    PROJECT = os.environ['GALILEO_PROJECT']
    LOG_STREAM = os.environ.get('GALILEO_LOG_STREAM', 'retrieval-agent')
    galileo_context.init(project=PROJECT, log_stream=LOG_STREAM)

''').lstrip() + FLUSH_MIDDLEWARE_SRC + textwrap.dedent('''
    app = FastAPI(title='Retrieval Agent')
    # ORDER MATTERS: last-added middleware is OUTERMOST. We want TracingMiddleware
    # outside so it sets the trace_id/parent_id contextvars before our flush
    # middleware runs inside it.
    app.add_middleware(FlushAfterRequestMiddleware)
    app.add_middleware(TracingMiddleware)

    KB = {
        'birthplace': ['Galileo Galilei was born in Pisa, Italy in 1564.'],
        'profession': ['Galileo Galilei taught geometry, mechanics, and astronomy at Padua.'],
        'research': ['Galileo built improved telescopes and observed the moons of Jupiter and the phases of Venus.'],
    }
    HINTS = {'work', 'company', 'location', 'education', 'research', 'study'}


    class Req(BaseModel):
        query: str


    class Resp(BaseModel):
        results: list[str]


    @log(span_type='retriever')
    def search(query: str) -> list[str]:
        q = query.lower()
        hits = []
        for cat, facts in KB.items():
            if cat in q or any(h in q for h in HINTS):
                hits.extend(facts)
        return hits[:3]


    @app.post('/retrieve', response_model=Resp)
    @log
    async def retrieve(r: Req):
        return Resp(results=search(r.query))


    @app.get('/health')
    async def health():
        return {'status': 'ok', 'agent': 'retrieval', 'project': PROJECT}


    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=int(os.environ['AGENT_PORT']), log_level='warning')
''')

ORCHESTRATOR_SRC = textwrap.dedent('''
    """Orchestrator agent — its OWN Galileo project & key."""
    import os
    from dotenv import load_dotenv
    import httpx
    from fastapi import FastAPI
    from pydantic import BaseModel

    load_dotenv()

    from galileo import galileo_context, get_tracing_headers, log, openai
    from galileo.middleware.tracing import TracingMiddleware

    PROJECT = os.environ['GALILEO_PROJECT']
    LOG_STREAM = os.environ.get('GALILEO_LOG_STREAM', 'orchestrator-agent')
    RETRIEVAL_URL = os.environ['RETRIEVAL_URL']
    MODEL = os.environ.get('LLM_MODEL', 'gpt-4o-mini')

    galileo_context.init(project=PROJECT, log_stream=LOG_STREAM)
    oai = openai.OpenAI()

''').lstrip() + FLUSH_MIDDLEWARE_SRC + textwrap.dedent('''
    app = FastAPI(title='Orchestrator Agent')
    app.add_middleware(FlushAfterRequestMiddleware)
    app.add_middleware(TracingMiddleware)


    class Req(BaseModel):
        question: str


    class Resp(BaseModel):
        answer: str
        trace_id: str | None
        outbound_tracing_headers: dict


    @log
    def analyze(q: str) -> dict:
        ql = q.lower()
        return {
            'needs_location': any(w in ql for w in ('where', 'location', 'city')),
            'needs_research': any(w in ql for w in ('research', 'study', 'discover')),
            'needs_work':     any(w in ql for w in ('work', 'employer', 'job')),
        }


    @app.post('/answer', response_model=Resp)
    @log
    async def answer(r: Req):
        _ = analyze(r.question)
        # Extract Galileo headers BEFORE the outbound call — these carry our trace_id
        # and our current span_id (so retrieval's spans nest under us).
        headers = get_tracing_headers()
        trace_id = headers.get('X-Galileo-Trace-ID')
        async with httpx.AsyncClient(base_url=RETRIEVAL_URL, timeout=30.0) as c:
            rresp = await c.post('/retrieve', json={'query': r.question}, headers=headers)
            rresp.raise_for_status()
            docs = rresp.json()['results']
        nl = chr(10)
        context = nl.join(f'- {d}' for d in docs) if docs else '(no documents retrieved)'
        sys_msg = 'Answer using only the context. If insufficient, say so.' + nl + 'Context:' + nl + context
        comp = oai.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': sys_msg},
                {'role': 'user', 'content': r.question},
            ],
        )
        return Resp(answer=comp.choices[0].message.content, trace_id=trace_id, outbound_tracing_headers=headers)


    @app.get('/health')
    async def health():
        return {'status': 'ok', 'agent': 'orchestrator', 'project': PROJECT}


    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=int(os.environ['AGENT_PORT']), log_level='warning')
''')

print(f'retrieval agent   : {len(RETRIEVAL_SRC.splitlines()):>3} lines, defined inline (no disk write)')
print(f'orchestrator agent: {len(ORCHESTRATOR_SRC.splitlines()):>3} lines, defined inline (no disk write)')
print(f'→ Step 3 will launch each via `python -c <source>` as a subprocess.')

retrieval agent   :  80 lines, defined inline (no disk write)
orchestrator agent:  96 lines, defined inline (no disk write)
→ Step 3 will launch each via `python -c <source>` as a subprocess.


## Step 3 — Launch both agents as real subprocesses

Each subprocess gets its own environment block:

| Variable | Per agent? |
|---|---|
| `GALILEO_API_KEY` | **Different** — drives which credential owns each span |
| `GALILEO_PROJECT` | **Same** — both agents write to the shared observability project |
| `GALILEO_LOG_STREAM` | **Same** — consolidates traces in one stream |
| `GALILEO_MODE=distributed` | Same — both agents trust incoming trace headers |
| `OPENAI_API_KEY` | Same — only the orchestrator uses it |

Each subprocess runs `uvicorn` pointed at its own service file. We use
`sys.executable` so the children pick up the parent's venv (same
`galileo`, `fastapi`, `openai` versions — no drift).

> **Why subprocesses, not threads?** Threads share `os.environ`, so two
> agents in one process can't carry *different* `GALILEO_API_KEY` values
> — the second `init()` silently overwrites the first's config.
> Subprocesses give each agent a fully isolated environment, mirroring a
> real cross-team deployment.

> **Why start them sequentially?** On a *cold* project, both agents'
> `galileo_context.init()` calls would race to create the project + log
> stream and one would return a 4xx conflict. Letting retrieval finish
> first guarantees the orchestrator's init takes the cheap `get()` path.


In [3]:
import subprocess
import time

import httpx

PY = sys.executable

def _agent_env(api_key: str, port: int, extra: dict | None = None) -> dict:
    """Build a subprocess env: per-agent key, shared project, shared log stream, agent's port."""
    env = os.environ.copy()
    env['GALILEO_API_KEY'] = api_key
    env['GALILEO_PROJECT'] = SHARED_PROJECT       # same for both agents
    env['GALILEO_LOG_STREAM'] = SHARED_LOG_STREAM # same for both agents
    env['GALILEO_MODE'] = 'distributed'
    env['AGENT_PORT'] = str(port)                 # read by uvicorn.run() inside each agent source
    for k in ('GALILEO_CONSOLE_URL', 'OPENAI_API_KEY'):
        if k in os.environ:
            env[k] = os.environ[k]
    env.update(extra or {})
    return env


RETRIEVAL_URL = f'http://127.0.0.1:{RETRIEVAL_PORT}'
ORCHESTRATOR_URL = f'http://127.0.0.1:{ORCHESTRATOR_PORT}'


def _wait_healthy(proc: subprocess.Popen, url: str, name: str, timeout: float = 30.0) -> dict:
    deadline = time.time() + timeout
    last_err = None
    while time.time() < deadline:
        try:
            r = httpx.get(f'{url}/health', timeout=2.0)
            if r.status_code == 200:
                return r.json()
        except httpx.HTTPError as e:
            last_err = e
        # Surface early subprocess crashes so we don't spin until timeout.
        if proc.poll() is not None:
            out = proc.stdout.read().decode('utf-8', errors='replace') if proc.stdout else ''
            raise RuntimeError(f'{name} exited with code {proc.returncode} before becoming healthy.\n--- stdout ---\n{out}')
        time.sleep(0.3)
    raise RuntimeError(f'{name} did not become healthy at {url} (last err: {last_err})')


# Idempotent re-run guard: terminate any leftover children from a prior run.
for var in ('_proc_retrieval', '_proc_orchestrator'):
    existing = globals().get(var)
    if existing is not None and existing.poll() is None:
        existing.terminate()
        try:
            existing.wait(timeout=5)
        except subprocess.TimeoutExpired:
            existing.kill()

# Launch retrieval FIRST and wait until it's healthy before starting the
# orchestrator. Why sequential? Each agent calls `galileo_context.init(...)` at
# module load, which does `Projects.get → Projects.create if missing` plus the
# same get-or-create for the log stream. If both subprocesses start in parallel
# on a *cold* project, both racing create() calls hit the API simultaneously and
# one returns 409/422 — which surfaces as a cryptic SDK parse error. Letting
# retrieval finish init first guarantees the project + log stream already exist
# when the orchestrator initializes, so its init() takes the cached get() path.
#
# We launch each agent with `python -c <SOURCE>` — the SOURCE strings from Step 2
# are passed directly as the script body. No files written to disk; the agent
# source lives only as Python strings in this notebook.
_proc_retrieval = subprocess.Popen(
    [PY, '-c', RETRIEVAL_SRC],
    env=_agent_env(RET_API_KEY, port=RETRIEVAL_PORT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
ret_health = _wait_healthy(_proc_retrieval, RETRIEVAL_URL, 'retrieval')

_proc_orchestrator = subprocess.Popen(
    [PY, '-c', ORCHESTRATOR_SRC],
    env=_agent_env(ORCH_API_KEY, port=ORCHESTRATOR_PORT, extra={'RETRIEVAL_URL': RETRIEVAL_URL}),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
orch_health = _wait_healthy(_proc_orchestrator, ORCHESTRATOR_URL, 'orchestrator')

print(f'retrieval    : {ret_health}')
print(f'orchestrator : {orch_health}')
print(f'→ both subprocesses are writing to project={SHARED_PROJECT!r}, log_stream={SHARED_LOG_STREAM!r}')

retrieval    : {'status': 'ok', 'agent': 'retrieval', 'project': 'GalileoEval_S19_CrossAgent'}
orchestrator : {'status': 'ok', 'agent': 'orchestrator', 'project': 'GalileoEval_S19_CrossAgent'}
→ both subprocesses are writing to project='GalileoEval_S19_CrossAgent', log_stream='cross-agent-trace'


## Step 4 — Drive the workflow, with the trace_id **defined in the notebook**

This is where the trace's lifecycle becomes visible. The notebook itself
is now a logging participant: it calls `galileo_context.init(...)`,
then `start_trace()` to generate the trace UUID and create the
corresponding `Trace` record on the backend, then reads
`get_tracing_headers()` and propagates them to the orchestrator on the
`POST /answer` call.

**Result:** the trace_id you see printed in this cell is the *same UUID*
the orchestrator will echo back, the *same UUID* retrieval's spans
attach to, and the *same UUID* that appears in the Galileo console.
The audience can literally watch one identifier flow through three
processes.

> **Why the notebook started it (not the orchestrator).** Galileo's
> backend creates Trace records via explicit `start_trace()`; spans then
> attach to a Trace by trace_id. In the original flow the orchestrator's
> `@log` auto-called `start_trace()`, so the orchestrator owned the
> trace. By moving that ownership to the notebook, we mirror what an
> upstream service (API gateway, frontend) would do in production — and
> we get a trace_id that's *already known* before any HTTP call happens.


In [4]:
from galileo import galileo_context, get_tracing_headers

# Make the notebook driver itself a Galileo logging participant.
# This is what lets us **define the trace_id before any HTTP call** — the
# notebook calls start_trace() (which creates the real Trace record in the
# backend), then propagates that trace_id via headers to the orchestrator.
# Orchestrator + retrieval are both downstream participants.
os.environ['GALILEO_MODE'] = 'distributed'
galileo_context.init(project=SHARED_PROJECT, log_stream=SHARED_LOG_STREAM)

QUESTIONS = [
    'What did Galileo Galilei research?',
    # 'Where did Galileo Galilei work?',
    # 'When was Galileo Galilei born?',
]

trace_ids: list[str] = []
for q in QUESTIONS:
    # ─── (1) Notebook generates and OWNS the trace_id ───
    notebook_logger = galileo_context.get_logger_instance()
    trace = notebook_logger.start_trace(input=q, name='notebook_demo_root')
    DEMO_TRACE_ID = str(trace.id)

    # ─── (2) Read the outbound tracing headers BEFORE any HTTP call ───
    outbound_headers = get_tracing_headers()

    print('=' * 70)
    print(f'Q: {q}')
    print()
    print(f'★ trace_id is born HERE, in the notebook (before any HTTP call):')
    print(f'    {DEMO_TRACE_ID}')
    print(f'  outbound headers the notebook will send to /answer:')
    for k, v in outbound_headers.items():
        print(f'    {k}: {v}')
    print()

    # ─── (3) Call the orchestrator, propagating the trace via headers ───
    r = httpx.post(f'{ORCHESTRATOR_URL}/answer', json={'question': q},
                   headers=outbound_headers, timeout=60.0)
    r.raise_for_status()
    data = r.json()
    trace_ids.append(data['trace_id'])

    print(f'A: {data["answer"]}')
    print()
    print(f'orchestrator echoed back trace_id it used:')
    print(f'    {data["trace_id"]}')
    print(f'  same UUID as the notebook generated? {data["trace_id"] == DEMO_TRACE_ID}')

    # ─── (4) Conclude + flush the notebook's view of the trace ───
    notebook_logger.conclude(output=data['answer'][:200])
    notebook_logger.flush()

print('\nAll trace IDs from this run (each generated in the notebook):')
for tid in trace_ids:
    print(f'  • {tid}')

Q: What did Galileo Galilei research?

★ trace_id is born HERE, in the notebook (before any HTTP call):
    f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc
  outbound headers the notebook will send to /answer:
    X-Galileo-Trace-ID: f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc
    X-Galileo-Parent-ID: f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc



A: Galileo Galilei researched geometry, mechanics, astronomy, and made significant observations using improved telescopes, including the moons of Jupiter and the phases of Venus.

orchestrator echoed back trace_id it used:
    f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc
  same UUID as the notebook generated? True



All trace IDs from this run (each generated in the notebook):
  • f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc


## Behind the scenes — how the trace gets stitched together

The three participants (notebook + orchestrator + retrieval) share **no
memory, no service discovery, no message bus** between them. The unified
trace you just saw exists entirely as **a UUID propagated through one
HTTP header** (with a second header naming the parent span). Here's the
lifecycle of the trace you just fired:

```
   ┌─ notebook (this Jupyter kernel) ────────────────────────────┐
   │                                                             │
   │  galileo_context.init(project, log_stream)                  │
   │  notebook_logger.start_trace(name="notebook_demo_root")     │
   │    →  trace_id  =  T  (fresh UUID, owned by the notebook)   │
   │    →  INGEST: the Trace record itself           ─────────┐  │
   │                                                          │  │
   │  headers = get_tracing_headers()                         │  │
   │    →  { X-Galileo-Trace-ID:  T,                          │  │
   │         X-Galileo-Parent-ID: T }                         │  │
   │                                                          │  │
   │  POST /answer  ──headers──┐                              │  │
   └───────────────────────────│──────────────────────────────│──┘
                               ▼                              │
   ┌─ orchestrator subprocess (KEY_A) ───────────────────────┐│
   │                                                         ││
   │  TracingMiddleware reads headers                        ││
   │    → contextvars: trace_id=T, parent_id=T               ││
   │                                                         ││
   │  @log answer():                                         ││
   │    singleton creates a NEW logger keyed by              ││
   │      (project, log_stream, T, T)                        ││
   │    _init_distributed_trace_stubs():                     ││
   │      stub Trace(id=T, name="stub_trace")                ││
   │      stub Span (id=T)                                   ││
   │    add span "answer"  → INGEST(trace=T, parent=T)  ─────┼┤
   │    add span "analyze" → INGEST(trace=T, parent=S_a) ────┼┤
   │                                                         ││
   │    headers = get_tracing_headers()                      ││
   │      → { X-Galileo-Trace-ID:  T,                        ││
   │          X-Galileo-Parent-ID: S_answer }                ││
   │                                                         ││
   │    POST /retrieve  ──headers──┐                         ││
   └───────────────────────────────│─────────────────────────│┘
                                   ▼                         │
   ┌─ retrieval subprocess (KEY_B) ─────────────────────────┐│
   │                                                        ││
   │  TracingMiddleware reads headers                       ││
   │    → contextvars: trace_id=T, parent_id=S_answer       ││
   │                                                        ││
   │  @log retrieve():                                      ││
   │    singleton creates a NEW logger keyed by             ││
   │      (project, log_stream, T, S_answer)                ││
   │    stub Trace(id=T, "stub_trace"), stub Span(id=S_a)   ││
   │    span "retrieve" → INGEST(trace=T, parent=S_a)  ─────┼┤
   │    @log search()   → INGEST(trace=T, parent=S_retr) ───┼┤
   │                                                        ││
   │  FlushAfter: wait until INGESTs ACKed                  ││
   └────────────────────────────────────────────────────────┘│
        │  (response back to orchestrator)                   │
        ▼                                                    │
   ┌─ orchestrator subprocess (continued) ──────────────────┐│
   │                                                        ││
   │    oai.chat.completions.create() →                     ││
   │      span "openai" (LLM) → INGEST(trace=T, parent=S_a)─┼┤
   │                                                        ││
   │    @log finalize: conclude span "answer"               ││
   │  FlushAfter: wait until INGESTs ACKed                  ││
   └────────────────────────────────────────────────────────┘│
        │  (response back to notebook)                       │
        ▼                                                    │
   notebook_logger.conclude(output=...)                      │
   notebook_logger.flush()                                   │
                                   ┌──── Galileo backend ────┴────┐
                                   │  Trace T  (one record)       │
                                   │   ├── answer    (S_a)        │
                                   │   ├── analyze                │
                                   │   ├── retrieve               │
                                   │   │    └── search            │
                                   │   └── openai (LLM)           │
                                   │  ↑ spans stitched by T       │
                                   └──────────────────────────────┘
```

### Four things worth pointing out

1. **The notebook *owns* the trace; orchestrator + retrieval *borrow*
   the UUID.** Only the notebook's `start_trace()` produces a real
   `Trace` row in the backend. Both subprocesses create local *stub*
   traces with the same UUID, marked `name="stub_trace"` — the SDK
   knows not to re-ingest. This avoids three services racing to define
   the same trace.

2. **Each process's logger is keyed by `(project, log_stream, trace_id,
   span_id)`.** Different inbound trace_ids land in different cached
   logger instances, so concurrent requests stay cleanly isolated. The
   *same* trace_id across three processes is exactly what makes those
   processes' spans logically attach to the same tree.

3. **The Galileo backend is the join point.** No process ever sees
   another's spans. They all send spans tagged with `trace_id=T` to
   Galileo, and the backend reassembles the tree at query time. That's
   why *"different keys, same org, same project"* works — the keys
   authenticate the **writes**, but the trace_id is what **associates**
   them.

4. **The trace_id is mortal but well-known the moment it's born.**
   `uuid.uuid4()` generates it in the notebook's `start_trace()`. Print
   it. Compare it to what the orchestrator echoes back. Click on the
   matching row in the Galileo console. All three are literally the
   same string. That's the entire "shared trace" story — no service
   mesh, no central allocator, just a 128-bit number passed around in
   HTTP headers.


## Step 5 — Open the single log stream

Because both agents wrote to the **same** project and log stream, the
request lands as **one trace** containing spans from both processes:

```
answer                  ← orchestrator subprocess (KEY_A)
├── analyze             ← orchestrator subprocess
├── retrieve            ← retrieval subprocess (KEY_B)
│     └── search        ← retriever span, retrieval subprocess
└── openai.completions  ← auto-logged LLM span
```

Open the log stream URL printed below, click the row, and all five spans
appear correctly nested — the **single pane of glass** the customer
asked for. Each span still records which API key produced it, so an
auditor can answer *"which agent wrote which span?"* without leaving the
console.


In [5]:
if trace_ids and trace_ids[-1] != '(none)':
    tid = trace_ids[-1]
    console = CONSOLE_URL.rstrip('/')
    print('Open the SHARED log stream in the Galileo console:')
    print(f'  {console}/projects?search={SHARED_PROJECT}')
    print()
    print(f'Most recent trace_id (search inside the log stream):')
    print(f'  {tid}')
    print()
    if DISTINCT_KEYS:
        print(f'Verification: this trace was written by TWO different keys')
        print(f'  • orchestrator spans → written with key …{ORCH_API_KEY[-6:]}')
        print(f'  • retrieval spans    → written with key …{RET_API_KEY[-6:]}')
        print(f'  • single project: {SHARED_PROJECT}')
        print(f'  • single log stream: {SHARED_LOG_STREAM}')
        print(f'→ This is the unified-view demo. Both agents are credentially')
        print(f'  isolated yet land in one consolidated trace tree.')
    else:
        print(f'Both agents used the SAME key (no GALILEO_API_KEY2 in .env).')
        print(f'→ Add GALILEO_API_KEY2 to .env and re-run to see the true')
        print(f'  "distinct keys, shared project" demo.')
else:
    print('No successful trace_ids captured this run.')

Open the SHARED log stream in the Galileo console:
  https://console.demo-v2.galileocloud.io/projects?search=GalileoEval_S19_CrossAgent

Most recent trace_id (search inside the log stream):
  f9fd7ed2-f6ce-4e00-a6e0-7358d1019adc

Verification: this trace was written by TWO different keys
  • orchestrator spans → written with key …-NHDSE
  • retrieval spans    → written with key …xcBRDY
  • single project: GalileoEval_S19_CrossAgent
  • single log stream: cross-agent-trace
→ This is the unified-view demo. Both agents are credentially
  isolated yet land in one consolidated trace tree.


## Step 6 — What this demo proves

### ✅ What we just demonstrated

- **Trace IDs are portable.** A UUID propagated as an HTTP header — works
  across processes, hosts, clouds, organizations.
- **Each agent owns its credentials.** Orchestrator and retrieval
  authenticate with *different* `GALILEO_API_KEY` values. No shared
  secrets between teams.
- **One unified trace tree.** Despite credential separation, every span
  for one request lands as **one trace** in **one log stream**. SREs
  open one row and see the entire distributed execution.
- **Span attribution survives.** Each span records the identity that
  wrote it. Auditors can still answer *"which service produced this?"*.

### 🚀 What this means for your setup

Your team operates inside a single Galileo org, so this is your direct
path: deploy each service with its own key, point every key at one
shared **platform observability** project + log stream, and propagate
the Galileo tracing headers on every internal HTTP call (or other
transport). Galileo handles the consolidation; you get credential
isolation per team and a single pane of glass for SREs.

> **If org topology ever changes** — e.g., a regulated region needs its
> own Galileo org — the foundation here (HTTP-header trace propagation)
> still applies. The only change is the destination: route spans
> through an OpenTelemetry collector that fans out per-org. See
> notebooks 10 / 11 / 12.


## Step 7 — Clean up

Terminate both subprocesses below. **No files were written** — the agent
source lives only as Python strings in this notebook, so there's nothing
to clean up on disk; killing the processes is the whole cleanup.

To delete the shared Galileo project entirely (recommended only after
you've explored it in the console), the simplest path is the **Galileo
console UI** — it handles permissions cleanly regardless of which key
created the project.


In [6]:
# SIGTERM both subprocesses, then give them up to 30s to atexit-flush any
# pending span ingests. Note: with the per-request FlushAfterRequestMiddleware
# installed in Step 2, all spans should already be in the backend by the time
# we get here — this longer timeout is defense in depth in case a subprocess
# crashed mid-handler and never got to flush.
for name, proc in [('orchestrator', _proc_orchestrator), ('retrieval', _proc_retrieval)]:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f'Stopped {name} (exit {proc.returncode}).')
    else:
        print(f'{name} already exited (code {proc.returncode}).')

# from galileo.projects import delete_project
# delete_project(name=ORCH_PROJECT)  # uses notebook's current API key
# delete_project(name=RET_PROJECT)

Stopped orchestrator (exit -15).
Stopped retrieval (exit -15).
